In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

In [2]:
print("Loading the Tuesday (Brute Force) Dataset")
file_path = "MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv"
df = pd.read_csv(file_path, low_memory=False)
df.columns = df.columns.str.strip()

Loading the Tuesday (Brute Force) Dataset


In [3]:
print("\n Traffic Distribution (Before Cleaning)")
#BENIGN, FTP-Patator and SSH-Patator
print(df['Label'].value_counts())


 Traffic Distribution (Before Cleaning)
Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64


In [4]:
df_clean = df.dropna()
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()

#Convert anything that isn't BENIGN to 1 (Attack). 
#i will group FTP-Patator and SSH-Patator together as a general "Brute Force" class.
df_clean['Label'] = df_clean['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

In [5]:
print("\n Splitting Data Before SMOTE")
X = df_clean.drop('Label', axis=1)
y = df_clean['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


 Splitting Data Before SMOTE


In [6]:
print("\n Applying SMOTE to balance the Training Data")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


 Applying SMOTE to balance the Training Data


In [7]:
print("\n Scaling the Data")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)


 Scaling the Data


In [8]:
print("\n Multi-Model Cycling")
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(50,), max_iter=100, random_state=42)
}

results = []


 Multi-Model Cycling


In [9]:
for name, model in models.items():
    print(f"  -> Training {name}...")
    start_time = time.time()
    
    model.fit(X_train_scaled, y_train_smote)
    y_pred = model.predict(X_test_scaled)
    
    train_time = time.time() - start_time
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 4),
        "Recall (Attack Catch Rate)": round(recall * 100, 4),
        "F1-Score": round(f1, 4),
        "Training Time (s)": round(train_time, 2)
    })

  -> Training Decision Tree...
  -> Training Random Forest...
  -> Training XGBoost...
  -> Training LightGBM...


c:\Users\Magda\Desktop\AnomalyDetection\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  -> Training Logistic Regression...
  -> Training MLP (Neural Net)...


In [10]:
print("\n Final Brute Force Leaderboard")
results_df = pd.DataFrame(results).sort_values(by="Recall (Attack Catch Rate)", ascending=False)
display(results_df)


 Final Brute Force Leaderboard


,Model,Accuracy (%),Recall (Attack Catch Rate),F1-Score,Training Time (s)
2,XGBoost,99.9978,100.0000,0.9996,16.51
0,Decision Tree,99.9989,99.9645,0.9998,38.24
1,Random Forest,99.9978,99.9645,0.9996,35.37
3,LightGBM,99.9966,99.9291,0.9995,21.85
4,Logistic Regression,98.6548,99.7162,0.8242,59.93
5,MLP (Neural Net),99.9237,99.7162,0.9880,648.29
